# NLP Evaluation, Hallucination, and Governance Lab


## Purpose

This lab simulates evaluation records for an NLP system.

The central point: language systems require evaluation beyond fluency.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
n = 1200

data = pd.DataFrame({
    "document_domain": rng.choice(["general", "technical", "legal"], size=n, p=[0.45, 0.35, 0.20]),
    "language_variety": rng.choice(["standard", "specialized", "informal"], size=n),
    "requires_citation": rng.choice([True, False], size=n, p=[0.60, 0.40]),
})

domain_risk = data["document_domain"].map({
    "general": 0.06,
    "technical": 0.12,
    "legal": 0.18,
})

variety_risk = data["language_variety"].map({
    "standard": 1.00,
    "specialized": 1.25,
    "informal": 1.10,
})

citation_risk = np.where(data["requires_citation"], 1.30, 1.00)

hallucination_probability = np.minimum(domain_risk * variety_risk * citation_risk, 0.90)

data["unsupported_claim"] = rng.binomial(1, hallucination_probability).astype(bool)

summary = (
    data
    .groupby(["document_domain", "language_variety"], as_index=False)
    .agg(
        sample_size=("unsupported_claim", "size"),
        unsupported_claim_rate=("unsupported_claim", "mean"),
    )
)

summary

In [ ]:
summary.to_csv("../outputs/notebook_nlp_hallucination_diagnostics.csv", index=False)
summary

## Governance Questions

- Which domains have the highest unsupported-claim rate?
- Are citations required for high-risk domains?
- Does the system distinguish retrieved evidence from generated prose?
- Is there human review for consequential use?
- How are errors logged and corrected?